# Create the training set

In [98]:
import random

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

nmt_train_set = []

for _ in range(10000):
    year = random.randint(1900, 2025)
    month = random.randint(1, 12)

    import calendar
    day = random.randint(1, calendar.monthrange(year, month)[1])

    source_date = f"{months[month-1]} {day:02d}, {year}"
    target_date = f"{year:04d}-{month:02d}-{day:02d}"

    nmt_train_set.append({
        "source_date": source_date,
        "target_date": target_date
    })

# Create the validation set

In [99]:
import random

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

nmt_valid_set = []

for _ in range(1000):
    year = random.randint(1900, 2025)
    month = random.randint(1, 12)

    import calendar
    day = random.randint(1, calendar.monthrange(year, month)[1])

    source_date = f"{months[month-1]} {day:02d}, {year}"
    target_date = f"{year:04d}-{month:02d}-{day:02d}"

    nmt_valid_set.append({
        "source_date": source_date,
        "target_date": target_date
    })

# Create the test set

In [100]:
import random

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

nmt_test_set = []

for _ in range(1000):
    year = random.randint(1900, 2025)
    month = random.randint(1, 12)

    import calendar
    day = random.randint(1, calendar.monthrange(year, month)[1])

    source_date = f"{months[month-1]} {day:02d}, {year}"
    target_date = f"{year:04d}-{month:02d}-{day:02d}"

    nmt_test_set.append({
        "source_date": source_date,
        "target_date": target_date
    })

# Create the tokenizer

In [101]:
def train_date_convert():
    for pair in nmt_train_set:
        yield pair["source_date"]
        yield pair["target_date"]


max_length = 256
vocab_size = 10000
import tokenizers

nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_date_convert(), nmt_tokenizer_trainer)


# Test the tokenizer

In [102]:
nmt_tokenizer.encode('September 30, 2006').ids

[97, 127, 4, 230]

# Create a utility class

In [103]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

# Create the data loaders

In [104]:
import torch
from torch.utils.data import DataLoader

def nmt_collate_fn(batch):
    src_dates = [pair['source_date'] for pair in batch]
    tgt_dates = [f"<s> {pair['target_date']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_dates)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_dates)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn)

# Create the model

In [105]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                 n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                              batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                              batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False)
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to("cuda")

# Define the training function

In [106]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

# Call the training function

In [107]:
n_epochs = 5
xentropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(nmt_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size).to("cuda")
history = train(nmt_model, optimizer, xentropy, accuracy, nmt_train_loader, nmt_valid_loader, n_epochs)

Epoch 1/5,                      train loss: 0.2814, train metric: 94.53%, valid metric: 100.00%
Epoch 2/5,                      train loss: 0.0012, train metric: 100.00%, valid metric: 100.00%
Epoch 3/5,                      train loss: 0.0005, train metric: 100.00%, valid metric: 100.00%
Epoch 4/5,                      train loss: 0.0003, train metric: 100.00%, valid metric: 100.00%
Epoch 5/5,                      train loss: 0.0002, train metric: 100.00%, valid metric: 100.00%


# Evaluate on the test set

In [108]:
evaluate_tm(nmt_model, nmt_test_loader, accuracy)

tensor(1., device='cuda:0')